# Text + Speech + EEG — Conversation-level PCA10 + XGBoost

Usa 05_text_speech_eeg.csv directamente. Conserva solo conversaciones completas y aplica StandardScaler + PCA10 a text y speech dentro del pipeline, evitando data leakage. EEG se escala dentro del pipeline y no se imputa.

In [1]:
# Solo necesario en Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5
N_COMPONENTS = 10

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_PATH = DATA_DIR / "05_text_speech_eeg.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"


In [ ]:
def load_partitions():
    """Carga directamente las particiones del paper."""
    partitions = pd.read_csv(PARTITIONS_PATH)
    return partitions[["subject_id", "avatar", "outer_fold"]].copy()

def get_metrics(y_true, y_pred, y_prob):
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def subject_level_predictions(pred_conv):
    """Agrega conversaciones por sujeto mediante la media de probabilidad DS."""
    pred_subject = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)["prob_1"]
        .mean()
    )
    pred_subject["pred"] = (pred_subject["prob_1"] >= 0.5).astype(int)
    return pred_subject


def summarize_mean_std(df, cols):
    return pd.concat([
        df[cols].mean().round(3).rename("mean"),
        df[cols].std().round(3).rename("std"),
    ], axis=1)


def make_pipeline(text_cols, speech_cols, eeg_cols):
    """PCA se ajusta dentro del pipeline: no usa datos de test."""
    preprocess = ColumnTransformer(
        transformers=[
            ("text_pca", Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=N_COMPONENTS, random_state=SEED)),
            ]), text_cols),
            ("speech_pca", Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=N_COMPONENTS, random_state=SEED)),
            ]), speech_cols),
            ("eeg", StandardScaler(), eeg_cols),
        ],
        remainder="drop",
    )

    return Pipeline([
        ("prep", preprocess),
        ("model", XGBClassifier()),
    ])


In [7]:
data = pd.read_csv(INPUT_PATH)
partitions = load_partitions()

# Identificación de modalidades.
text_cols = sorted([c for c in data.columns if c.startswith("text_")], key=lambda c: int(c.split("_", 1)[1]))
speech_cols = sorted([c for c in data.columns if c.startswith("speech_")], key=lambda c: int(c.split("_", 1)[1]))
meta_cols = {"subject_id", "avatar", "label"}
eeg_cols = [c for c in data.columns if c not in meta_cols and c not in text_cols and c not in speech_cols]
feature_cols = text_cols + speech_cols + eeg_cols

# Añadir outer_fold y conservar solo conversaciones con las tres modalidades completas.
data["subject_id"] = data["subject_id"].astype(str).str.strip().str.upper()
data["avatar"] = data["avatar"].astype(str).str.strip()
partitions = shuffle(partitions, random_state=SEED).reset_index(drop=True)

df = data.merge(partitions, on=["subject_id", "avatar"], how="inner")

n_before = len(df)
df = df.dropna(subset=feature_cols).copy()
n_removed = n_before - len(df)

print("Filas iniciales con partición:", n_before)
print("Filas eliminadas por falta de alguna modalidad:", n_removed)
print("Filas trimodales finales:", len(df))
print("Sujetos finales:", df["subject_id"].nunique())
print("Text features originales:", len(text_cols), "-> PCA", N_COMPONENTS)
print("Speech features originales:", len(speech_cols), "-> PCA", N_COMPONENTS)
print("EEG features:", len(eeg_cols))

print("\nSujetos por outer fold:")
display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))

print("\nDistribución de clases por outer fold:")
display(pd.crosstab(df.drop_duplicates("subject_id")["outer_fold"], df.drop_duplicates("subject_id")["label"]))


Filas iniciales con partición: 600
Filas eliminadas por falta de alguna modalidad: 42
Filas trimodales finales: 558
Sujetos finales: 94
Text features originales: 768 -> PCA 10
Speech features originales: 1024 -> PCA 10
EEG features: 27

Sujetos por outer fold:


,n_subjects
outer_fold,
1,20
2,18
3,18
4,19
5,19



Distribución de clases por outer fold:


label,0,1
outer_fold,,
1,11,9
2,10,8
3,11,7
4,12,7
5,11,8


In [8]:
param_grid = {
    "model__max_depth": list(range(3, 12)),
    "model__n_estimators": [25, 50, 100, 200],
}

scoring = {
    "WAcc": "accuracy",
    "UAcc": "balanced_accuracy",
    "auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

metric_cols = ["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]


In [9]:
OUT_DIR = DATA_DIR / "07_results_text_speech_eeg_conversation_level_PCA10"
OUT_DIR.mkdir(parents=True, exist_ok=True)

conv_metrics_rows = []
subject_metrics_rows = []
best_params_rows = []
all_conv_predictions = []

for fold in sorted(df["outer_fold"].unique()):
    print(f"\n===== OUTER FOLD {fold} =====")

    dev = df[df["outer_fold"] != fold].reset_index(drop=True)
    test = df[df["outer_fold"] == fold].reset_index(drop=True)

    X_dev = dev[feature_cols]
    y_dev = dev["label"].to_numpy(dtype=int)
    groups_dev = dev["subject_id"].to_numpy()

    X_test = test[feature_cols]
    y_test = test["label"].to_numpy(dtype=int)

    inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    grid = GridSearchCV(
        estimator=make_pipeline(text_cols, speech_cols, eeg_cols),
        param_grid=param_grid,
        scoring=scoring,
        refit="UAcc",
        cv=inner_cv,
        n_jobs=-1,
        verbose=0,
    )
    grid.fit(X_dev, y_dev, groups=groups_dev)

    prob_1 = grid.best_estimator_.predict_proba(X_test)[:, 1]
    pred = (prob_1 >= 0.5).astype(int)

    pred_conv = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
    pred_conv["prob_1"] = prob_1
    pred_conv["pred"] = pred
    all_conv_predictions.append(pred_conv)

    best_idx = grid.best_index_
    conv_metrics = get_metrics(y_test, pred, prob_1)
    conv_metrics["outer_fold"] = fold
    conv_metrics["cv_f1"] = grid.cv_results_["mean_test_f1"][best_idx]
    conv_metrics_rows.append(conv_metrics)

    pred_subject = subject_level_predictions(pred_conv)
    subject_metrics = get_metrics(pred_subject["label"], pred_subject["pred"], pred_subject["prob_1"])
    subject_metrics["outer_fold"] = fold
    subject_metrics_rows.append(subject_metrics)

    best_params_rows.append({
        "outer_fold": fold,
        "best_max_depth": grid.best_params_["model__max_depth"],
        "best_n_estimators": grid.best_params_["model__n_estimators"],
        "best_inner_UAcc": grid.cv_results_["mean_test_UAcc"][best_idx],
        "best_inner_f1": grid.cv_results_["mean_test_f1"][best_idx],
    })

    print("Best params:", grid.best_params_)
    print("Conversation-level CV F1:", round(grid.cv_results_["mean_test_f1"][best_idx], 3))
    print("Conversation-level Test F1:", round(conv_metrics["f1"], 3))
    print("Subject-level Test F1:", round(subject_metrics["f1"], 3))

conv_metrics_df = pd.DataFrame(conv_metrics_rows)
subject_metrics_df = pd.DataFrame(subject_metrics_rows)
best_params_df = pd.DataFrame(best_params_rows)
conv_predictions_df = pd.concat(all_conv_predictions, ignore_index=True)
subject_predictions_global = subject_level_predictions(conv_predictions_df)
global_subject_metrics = get_metrics(
    subject_predictions_global["label"],
    subject_predictions_global["pred"],
    subject_predictions_global["prob_1"],
)

results_summary = pd.DataFrame({
    "metric": ["Conversation-level CV F1", "Conversation-level Test F1", "Subject-level Test F1"],
    "mean": [conv_metrics_df["cv_f1"].mean(), conv_metrics_df["f1"].mean(), subject_metrics_df["f1"].mean()],
    "std": [conv_metrics_df["cv_f1"].std(), conv_metrics_df["f1"].std(), subject_metrics_df["f1"].std()],
}).round(3)

conv_metrics_df.to_csv(OUT_DIR / "conversation_level_outer_metrics.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "subject_level_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "best_params_by_outer_fold.csv", index=False)
conv_predictions_df.to_csv(OUT_DIR / "conversation_predictions.csv", index=False)
subject_predictions_global.to_csv(OUT_DIR / "subject_predictions_global.csv", index=False)
results_summary.to_csv(OUT_DIR / "main_results_summary.csv", index=False)

print("\nResultados principales")
display(results_summary)

print("\nMétricas subject-level globales")
display(pd.Series(global_subject_metrics).round(3).to_frame("global"))

print("\nArchivos guardados en:", OUT_DIR)



===== OUTER FOLD 1 =====
Best params: {'model__max_depth': 7, 'model__n_estimators': 100}
Conversation-level CV F1: 0.504
Conversation-level Test F1: 0.49
Subject-level Test F1: 0.5

===== OUTER FOLD 2 =====
Best params: {'model__max_depth': 4, 'model__n_estimators': 200}
Conversation-level CV F1: 0.529
Conversation-level Test F1: 0.442
Subject-level Test F1: 0.308

===== OUTER FOLD 3 =====
Best params: {'model__max_depth': 3, 'model__n_estimators': 50}
Conversation-level CV F1: 0.479
Conversation-level Test F1: 0.588
Subject-level Test F1: 0.714

===== OUTER FOLD 4 =====
Best params: {'model__max_depth': 3, 'model__n_estimators': 25}
Conversation-level CV F1: 0.509
Conversation-level Test F1: 0.597
Subject-level Test F1: 0.667

===== OUTER FOLD 5 =====
Best params: {'model__max_depth': 9, 'model__n_estimators': 100}
Conversation-level CV F1: 0.472
Conversation-level Test F1: 0.494
Subject-level Test F1: 0.5

Resultados principales


,metric,mean,std
0,Conversation-level CV F1,0.499,0.023
1,Conversation-level Test F1,0.522,0.068
2,Subject-level Test F1,0.538,0.161



Métricas subject-level globales


,global
WAcc,0.670
UAcc,0.640
auc,0.717
f1,0.537
precision,0.643
recall,0.462
kappa,0.292



Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/07_results_text_speech_eeg_conversation_level_PCA10
